# 03 · Factores, escenarios y riesgo de la posición inicial

Muestra la capa (iii) (`src/optimum/risk.py`): cambios de factores, escenarios a un año (bootstrap por bloques + estrés), matriz de resultados $r_{k,s}$ por revalorización completa y VaR/CVaR de la posición inicial. Formulación §5 y §7; decisiones D-06, D-09 v3, D-11, D-14, D-15 y D-16.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from optimum import risk
from optimum.io.json_contract import load_input
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

doc = load_input()
inst = pd.DataFrame(doc["instruments"]).set_index("id")

## Cambios de factores

Desde `marketHistory` de input.json: Δ absoluta en tasas, log-retorno en FX y equity.

In [ ]:
chg = risk.factor_changes(doc)
fig, ax = new_figure(width=0.8, aspect=0.85)
corr = chg.corr()
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)), corr.index, fontsize=6)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlación de cambios mensuales de factores")
save_fig(fig, "correlacion_factores")

## Escenarios a un año

- **Bootstrap:** `bootstrapCount` escenarios; cada uno suma 2 ventanas centradas de 6 meses (D-09 v3) más la deriva anual μ (D-06).
- **Estrés:** 4 escenarios de `stressScenarios`, sin deriva (D-11, D-16).
- $p_s$ = `bootstrapProbability` / $N_B$ para el bootstrap y `probability` para cada estrés.

In [ ]:
scen = risk.build_scenarios(doc)
kind = np.asarray(scen.kind)
mu = risk.drift(doc, chg)
print(
    f"{len(scen.ids)} escenarios: {(kind == 'bootstrap').sum()} bootstrap + {(kind == 'stress').sum()} estrés; "
    f"Σp = {scen.probs.sum():.6f}, p(estrés) = {scen.probs[kind == 'stress'].sum():.4f}"
)
mu[mu != 0].rename("μ anual")

### Controles

- σ del shock bootstrap a 12 m frente a la σ histórica de la suma móvil de 12 meses: deben ser del mismo orden. Con cambios persistentes, σ mensual·√12 subestimaría el riesgo anual (tabla de persistencia del notebook 00).
- Media del bootstrap frente a μ: tras centrar las ventanas, solo difieren por ruido de muestreo.
- Tasa mínima por escenario: no se aplica piso (§7.3); se reporta como control.

In [ ]:
diag = risk.scenario_diagnostics(doc, scen)
boot = scen.deltas[kind == "bootstrap"]
key = ["PEN_SPOT_ON", "PEN_SPOT_10Y", "USD_SPOT_ON", "USD_SPOT_10Y", "FX_PENUSD", "EQUITY_USD"]
sig = (
    diag["sigma"]
    .loc[key]
    .assign(**{"σ mensual·√12": np.sqrt(12) * chg[key].std(), "media boot.": boot[key].mean(), "μ": mu[key]})
)
sig["razón boot./hist."] = sig["bootstrap_12m"] / sig["historical_12m"]
save_table(
    sig.rename(columns={"bootstrap_12m": "σ bootstrap 12 m", "historical_12m": "σ histórica 12 m"}),
    "sigma_escenarios",
    caption="Dispersión y media de los shocks a un año por factor (decimales).",
    label="tab:sigma-escenarios",
    fmt="{:.4f}",
)
sig.round(4)

In [ ]:
mr = diag["min_rate"]
print(
    f"Tasa spot mínima: {100 * mr.min():.2f} % ({mr.idxmin()}); escenarios con alguna tasa < 0: {(mr < 0).sum()}"
)
mr[kind == "stress"].mul(100).round(2).rename("tasa mínima (%)")

### Escenarios de estrés (Δ a un año)

In [ ]:
stress = scen.deltas[kind == "stress"]
spot = [f"{c}_SPOT_{n}" for c in ["PEN", "USD"] for n in ["ON", "1Y", "3Y", "5Y", "10Y", "20Y"]]
tab_stress = pd.concat(
    [
        1e4 * stress[spot],
        100 * (np.exp(stress[["FX_PENUSD", "EQUITY_USD"]]) - 1),
    ],
    axis=1,
).T
tab_stress.index = [f"{f} (pb)" for f in spot] + ["FX (%)", "Equity (%)"]
save_table(
    tab_stress,
    "escenarios_estres",
    caption="Escenarios de estrés del Caso C: shifts de la curva spot (pb) y variaciones de FX y equity (\\%).",
    label="tab:estres",
    fmt="{:,.1f}",
)
tab_stress.round(1)

### Forwards implícitas en cada estrés (D-11 v2)

En la transición se interpola $t\,s(t)$, así que la forward cambia parejo entre los nodos fijos. El cambio medio lo fijan los datos: en C_STRESS_3, las forwards entre 1Y y 5Y caen unos −37 pb aunque todas las spot suben.

In [ ]:
from optimum.valuation import base_state

state0 = base_state(doc)
fwd_stress = pd.DataFrame(
    {
        sid: {k: 1e4 * (v - state0.forwards[k]) for k, v in state0.shocked(row.to_dict()).forwards.items()}
        for sid, row in stress.iterrows()
    }
)
fwd_stress.index = [f"{k} (pb)" for k in fwd_stress.index]
save_table(
    fwd_stress,
    "forwards_estres",
    caption="Cambio de las forwards implícitas en cada estrés (pb).",
    label="tab:forwards-estres",
    fmt="{:,.1f}",
)
fwd_stress.round(1)

In [ ]:
fig, axes = new_figure(2, 2, aspect=0.7)
panels = [
    ("PEN_SPOT_ON", "PEN ON (pb)", 1e4, COLORS["PEN"]),
    ("USD_SPOT_ON", "USD ON (pb)", 1e4, COLORS["USD"]),
    ("FX_PENUSD", "FX (log-ret., %)", 100, COLORS["USD"]),
    ("EQUITY_USD", "Equity (log-ret., %)", 100, COLORS["EQUITY"]),
]
for ax, (f, label, k, color) in zip(axes.flat, panels, strict=True):
    ax.hist(k * boot[f], bins=30, color=color, alpha=0.7)
    for v in k * stress[f]:
        ax.axvline(v, color="black", ls="--", lw=0.8)
    ax.set_title(label)
axes[0, 0].set_ylabel("Escenarios")
axes[1, 0].set_ylabel("Escenarios")
fig.suptitle("Shocks a un año: bootstrap (histograma) y estrés (líneas)", fontsize=10)
fig.tight_layout()
save_fig(fig, "shocks_escenarios")

## Matriz de resultados $r_{k,s}$

$r_{k,s} = (V_k(t_H) + CF_k(t_0, t_H]) / V_k(t_0) - 1$ por revalorización completa. No depende del tamaño de la posición, así que el optimizador puede tratarla como dato (LP).

In [ ]:
R = risk.scenario_returns(doc, scen.deltas)  # ≈ 45 s
p = scen.probs
er = pd.DataFrame(
    {
        "lado": inst["side"],
        "moneda": inst["currency"],
        "E[r] (%)": 100 * (R @ p),
        "σ(r) (%)": 100 * np.sqrt(((R.sub(R @ p, axis=0)) ** 2) @ p),
        "r estrés mín. (%)": 100 * R.loc[:, kind == "stress"].min(axis=1),
        "r estrés máx. (%)": 100 * R.loc[:, kind == "stress"].max(axis=1),
    }
)
save_table(
    er,
    "rendimientos_escenarios",
    caption="Resultado a un año por instrumento en los escenarios (para pasivos, costo).",
    label="tab:rendimientos",
)
er.round(2)

## Riesgo de la posición inicial

$\Delta PN_s = \sum_i x^0_i r_{i,s} - \sum_j y^0_j r_{j,s}$, pérdida $L_s = -\Delta PN_s$. VaR y CVaR al 95 % con átomo fraccional (§5). Es el benchmark del optimizador.

In [ ]:
x0, sides = inst["market_value_pen"], inst["side"]
dpn = risk.pn_change(R, x0, sides)
alpha = doc["caseParameters"]["cvarAlpha"]
var, cvar = risk.var_cvar(-dpn, alpha, p)
pn0 = x0[sides == "ASSET"].sum() - x0[sides == "LIABILITY"].sum()
e_dpn = p @ dpn
tail = -dpn >= var
metrics = pd.Series(
    {
        "PN0": pn0,
        "E[ΔPN]": e_dpn,
        "E[PN_T]": pn0 + e_dpn,
        "σ(ΔPN)": np.sqrt(p @ (dpn - e_dpn) ** 2),
        f"VaR {alpha:.0%}": var,
        f"CVaR {alpha:.0%}": cvar,
        "P(PN_T < 120)": p[pn0 + dpn < doc["constraints"]["common"]["minNetWorth"]].sum(),
        "prob. cola: estrés / total": p[tail & (kind == "stress")].sum() / p[tail].sum(),
    }
)
for sid in scen.ids[-4:]:
    metrics[f"ΔPN {sid}"] = dpn[scen.ids.index(sid)]
save_table(
    metrics.to_frame("valor"),
    "riesgo_posicion_inicial",
    caption="Riesgo a un año de la posición inicial (S/ mm salvo probabilidades).",
    label="tab:riesgo-inicial",
)
metrics.round(3)

In [ ]:
fig, ax = new_figure(aspect=0.45)
b = kind == "bootstrap"
ax.hist(dpn[b], bins=40, color=COLORS["EQUITY"], alpha=0.7, label="Bootstrap")
ax.axvline(-var, color="black", lw=1, label=f"−VaR {alpha:.0%}")
ax.axvline(-cvar, color="black", ls=":", lw=1.2, label=f"−CVaR {alpha:.0%}")
ax.scatter(
    dpn[~b], np.zeros((~b).sum()), marker="v", s=40, color=COLORS["LIABILITY"], zorder=3, label="Estrés"
)
ax.set(xlabel="ΔPN a un año (S/ mm)", ylabel="Escenarios", title="Posición inicial: distribución de ΔPN")
ax.legend()
save_fig(fig, "distribucion_dpn_inicial")

## Coherencia: método local vs. revalorización completa (D-14)

$M$ (instrumentos × factores) por diferencias finitas centrales sobre `value_at_t0`; $\sigma_{local} = \sqrt{w^\top M \Sigma M^\top w}$ con $w$ = +posición en activos, −posición en pasivos, en S/ mm. Se usan dos $\Sigma$: la mensual ×12 (i.i.d.) y la de los shocks bootstrap a 12 m. Las forwards no tienen sensibilidad propia porque se recalculan desde la spot (D-10).

In [ ]:
from optimum.valuation import calibrate_spreads, value_at_t0

spreads = calibrate_spreads(doc)
factors = [f for f in chg.columns if "_FWD_" not in f]
M = risk.sensitivities(lambda d: value_at_t0(doc, spreads, d), factors)
w = np.where(sides.reindex(M.index) == "ASSET", 1.0, -1.0)  # M ya está en S/ mm por unidad de factor
g = w @ M.to_numpy()  # sensibilidad del PN a cada factor
sigma_iid = risk.covariance(chg[factors]).to_numpy()
sigma_boot = boot[factors].cov().to_numpy()
coh = pd.Series(
    {
        "local, Σ mensual × 12 (i.i.d.)": np.sqrt(g @ sigma_iid @ g),
        "local, Σ bootstrap 12 m": np.sqrt(g @ sigma_boot @ g),
        "revalorización completa (solo bootstrap)": np.sqrt(np.cov(dpn[b])),
    },
    name="σ(ΔPN) S/ mm",
)
coh.round(2)

## Lectura

*(Completar tras revisar las salidas: exposición dominante, peso del estrés en la cola, coherencia local vs. completa y por qué la σ i.i.d. es menor.)*